In [1]:
import numpy as np
import pandas as pd

In [2]:
rng = np.random.default_rng(seed=42)
N_PATIENTS = 2000

In [3]:
print("Pronto! numpy", np.__version__, "| pandas", pd.__version__)

Pronto! numpy 2.5.4 | pandas 3.1.1


In [4]:
CLASS_PROPORTIONS = {
    "nociceptive": 0.40,
    "neuropathic": 0.15,
    "nociplastic": 0.20,
    "mixed": 0.25,
}

labels = rng.choice(
    list(CLASS_PROPORTIONS.keys()),
    size=N_PATIENTS,
    p=list(CLASS_PROPORTIONS.values()),
)

df = pd.DataFrame({
    "patient_id": range(1, N_PATIENTS + 1),
    "mechanism": labels,
})

df["mechanism"].value_counts(normalize=True).round(3)

mechanism
nociceptive    0.394
mixed          0.262
nociplastic    0.191
neuropathic    0.152
Name: proportion, dtype: float64

In [5]:
DURATION_PARAMS = {
    "nociceptive": (2, 0.9),
    "neuropathic": (6, 0.8),
    "nociplastic": (24, 0.7),
    "mixed": (12, 0.8),
}

duration = np.empty(N_PATIENTS)

for mechanism, (median, spread) in DURATION_PARAMS.items():
    mask = df["mechanism"] == mechanism
    duration[mask] = rng.lognormal(mean=np.log(median), sigma=spread, size=mask.sum())

df["pain_duration_months"] = np.clip(duration, 0.25, 240).round(1)

is_nociplastic = df["mechanism"] == "nociplastic"
df.loc[is_nociplastic, "pain_duration_months"] = df.loc[is_nociplastic, "pain_duration_months"].clip(lower=3)

df.groupby("mechanism")["pain_duration_months"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
mechanism,,,,,,,,
mixed,524.0,16.9,15.9,1.5,6.9,12.1,20.4,116.9
neuropathic,305.0,8.8,8.4,0.7,3.6,6.1,11.1,69.4
nociceptive,789.0,3.0,3.3,0.2,1.1,2.1,3.6,26.2
nociplastic,382.0,30.1,25.0,3.5,15.0,24.1,37.2,208.5
